# Scenario 16: Quote-To-Cash (Group Chat)

| Field | Value |
| --- | --- |
| Scenario id | `scenario-16-quote-to-cash-group-chat` |
| Pattern | `group-chat` |
| API | `Responses API` |

**Learning goal:** Learn collaborative quote review where agents debate product fit, pricing risk, legal terms, and final quote readiness until they converge.

> Use Group Chat when a visible debate and cross-check between roles improves the quote before it is finalized.


In [ ]:
import os

from IPython.display import HTML, display


_APTOS_STYLE = """
<style>
:root { --jp-content-font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif; }
.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .rendered_html, .jp-OutputArea-output {
    font-family: 'Aptos', 'Segoe UI', 'Helvetica Neue', sans-serif;
    line-height: 1.55;
}
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3 {
    font-family: 'Aptos Display', 'Aptos', 'Segoe UI', sans-serif;
    font-weight: 600;
}
</style>
"""


def apply_notebook_style() -> str:
    display(HTML(_APTOS_STYLE))
    return _APTOS_STYLE


OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:14b")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

apply_notebook_style()
print(f"Ollama target: {OLLAMA_HOST} / {OLLAMA_MODEL}")


## Pattern: Group Chat

Group chat orchestration creates a visible multi-agent discussion. A selector function chooses the next participant and a termination function decides when the discussion is good enough.

Best fit: decisions that benefit from critique, tradeoffs, and a short transcript.

## API Shape

**Responses API -- startup-selected scenario shape.** The scenario and orchestration pattern are wired in at server start. Each client request uses the standard OpenAI-compatible `/responses` body -- a plain chat-style input. The client never specifies which agents run; the server owns the orchestration entirely.

This is a Scenario 16 quote-to-cash variant. The same six business roles (CRM trigger, customer context, SKU discovery, product fit, pricing and legal, quote generation) appear in every Scenario 16 notebook -- only the orchestration pattern changes. Compare notebooks 16a-16e to see how the same roles behave under sequential, concurrent, handoff, group-chat, and magentic coordination.

## Pattern Anatomy

| Dimension | Detail |
| --- | --- |
| Control flow | Agents take turns until a termination condition is met. |
| Coordination | A selector function chooses the next participant. |
| Output | The transcript shows critique, refinement, and convergence. |
| Best when | Use when visible debate improves the answer. |

## Coded Agents

| Agent | Role | Tools |
| --- | --- | --- |
| `QuoteTriggerAgent` | Monitors CRM for quote-trigger conditions. | `note_observation`, `make_checklist` · MCP: `crm_get_quote_trigger` |
| `CustomerContextAgent` | Enriches the customer profile from CRM. | `note_observation`, `compose_summary` · MCP: `crm_get_customer_profile` |
| `SkuDiscoveryAgent` | Finds SKUs, bundles, and catalog entries. | `note_observation`, `make_checklist` · MCP: `product_search_catalog` |
| `ProductFitAgent` | Validates product compatibility and availability. | `note_observation`, `make_checklist` · MCP: `product_validate_skus` |
| `PricingTermsAgent` | Resolves pricing, finance, and legal terms. | `note_observation`, `rate_risk` · MCP: `pricing_calculate_quote`, `legal_evaluate_terms` |
| `QuoteGenerationAgent` | Generates the final customer-ready quote package. | `compose_summary`, `extract_action_items` · MCP: `quote_format_package` |

> **Instructor note:** Every agent receives deterministic Python tool functions.
> Tools labelled _role defaults_ are assigned by keyword matching on the agent name
> and description. MCP tools map to the inlined context functions in the cell below.


In [ ]:
from collections.abc import Callable, Sequence


_RISK_TIERS: tuple[tuple[int, str], ...] = (
    (20, "critical"),
    (12, "high"),
    (6, "medium"),
    (0, "low"),
)


def note_observation(category: str, observation: str) -> str:
    """Record a single structured observation."""

    return f"[{category.strip().lower()}] {observation.strip()}"


def rate_risk(impact: int, likelihood: int) -> dict[str, object]:
    """Score a risk deterministically from impact and likelihood."""

    impact_v = max(1, min(5, int(impact)))
    likelihood_v = max(1, min(5, int(likelihood)))
    score = impact_v * likelihood_v
    tier = next(name for floor, name in _RISK_TIERS if score >= floor)
    return {"impact": impact_v, "likelihood": likelihood_v, "score": score, "tier": tier}


def make_checklist(items: Sequence[str]) -> str:
    """Render a Markdown checklist from a list of items."""

    cleaned = [str(item).strip() for item in items if str(item).strip()]
    if not cleaned:
        return "- [ ] (no items)"
    return "\n".join(f"- [ ] {item}" for item in cleaned)


def extract_action_items(text: str) -> list[str]:
    """Extract concise action items from free text."""

    raw: list[str] = []
    for line in (text or "").replace(";", "\n").splitlines():
        for piece in line.split(". "):
            candidate = piece.strip(" -*\t.").strip()
            if 0 < len(candidate) <= 160:
                raw.append(candidate)
    seen: set[str] = set()
    ordered: list[str] = []
    for item in raw:
        key = item.lower()
        if key not in seen:
            seen.add(key)
            ordered.append(item)
    return ordered[:8]


def tally_votes(votes: Sequence[str]) -> dict[str, object]:
    """Tally approve, reject, and abstain style votes."""

    counts = {"approve": 0, "reject": 0, "abstain": 0}
    for vote in votes:
        text = str(vote).lower()
        if any(token in text for token in ("reject", "no-go", "block", "deny", "veto")):
            counts["reject"] += 1
        elif any(token in text for token in ("approve", "yes", "go ahead", "go-ahead")):
            counts["approve"] += 1
        else:
            counts["abstain"] += 1
    if counts["approve"] > counts["reject"]:
        decision = "approved"
    elif counts["reject"] > counts["approve"]:
        decision = "rejected"
    else:
        decision = "undecided"
    return {"counts": counts, "decision": decision}


def compose_summary(sections: dict[str, str]) -> str:
    """Compose a readable summary from a mapping of heading to content."""

    if not sections:
        return "No content to summarize."
    return "\n\n".join(f"## {heading}\n{content.strip()}" for heading, content in sections.items())


CODE_TOOLS: dict[str, Callable[..., object]] = {
    "note_observation": note_observation,
    "rate_risk": rate_risk,
    "make_checklist": make_checklist,
    "extract_action_items": extract_action_items,
    "tally_votes": tally_votes,
    "compose_summary": compose_summary,
}

_ROLE_TOOL_RULES: tuple[tuple[tuple[str, ...], tuple[str, ...]], ...] = (
    (("risk", "fraud", "security", "threat", "priority", "incident"), ("note_observation", "rate_risk")),
    (("board", "chair", "council", "panel", "vote", "debate"), ("note_observation", "tally_votes")),
    (("plan", "manager", "trigger", "intake", "coordinat", "orchestrat", "scope"), ("note_observation", "make_checklist")),
    (("summary", "editor", "writer", "docs", "comms", "generation", "packet", "report", "action", "quote"), ("compose_summary", "extract_action_items")),
    (("validate", "fit", "check", "audit", "review", "evidence", "compat", "depend"), ("note_observation", "make_checklist")),
)


def default_code_tools_for(name: str, description: str = "") -> tuple[str, ...]:
    text = f"{name} {description}".lower()
    for keywords, tools in _ROLE_TOOL_RULES:
        if any(keyword in text for keyword in keywords):
            return tools
    return ("note_observation", "compose_summary")


def effective_code_tools(spec: object) -> tuple[str, ...]:
    explicit = tuple(getattr(spec, "code_tools", ()) or ())
    if explicit:
        return explicit
    return default_code_tools_for(getattr(spec, "name", ""), getattr(spec, "description", ""))


def resolve_code_tools(names: Sequence[str]) -> list[Callable[..., object]]:
    resolved: list[Callable[..., object]] = []
    for name in names:
        try:
            resolved.append(CODE_TOOLS[name])
        except KeyError as exc:
            raise ValueError(f"Unknown code tool '{name}'.") from exc
    return resolved


MCP_TOOL_FUNCTIONS: dict[str, Callable[..., object]] = {}


## MCP Tool Context

In production, these quote-to-cash context functions are exposed by a local FastMCP stdio server and attached to
agents with `MCPStdioTool` using per-agent allowed tools. This notebook inlines the same deterministic
functions as plain function tools so it remains standalone.


In [ ]:
import hashlib
from typing import Any


_QUOTE_TRIGGERS: dict[str, dict[str, Any]] = {
    "OPP-5001": {
        "opportunity_id": "OPP-5001",
        "account_id": "ACC-3300",
        "stage": "Negotiation",
        "quote_ready": True,
        "trigger_conditions": [
            "Opportunity stage is Negotiation or later.",
            "Primary contact and billing account are set.",
            "Budget is confirmed by the customer.",
        ],
        "blocking_conditions": [],
    },
    "OPP-5002": {
        "opportunity_id": "OPP-5002",
        "account_id": "ACC-3301",
        "stage": "Discovery",
        "quote_ready": False,
        "trigger_conditions": ["Opportunity stage is Negotiation or later."],
        "blocking_conditions": [
            "Opportunity is still in Discovery.",
            "No confirmed budget on the opportunity.",
        ],
    },
}

_CUSTOMER_PROFILES: dict[str, dict[str, Any]] = {
    "ACC-3300": {
        "account_id": "ACC-3300",
        "customer_name": "Contoso Manufacturing",
        "address": "120 Industrial Way, Aurora, IL 60502, USA",
        "msa_status": "signed",
        "account_status": "active",
        "segment": "enterprise",
        "buying_context": "Expanding plant automation; standardizing on one analytics platform.",
    },
    "ACC-3301": {
        "account_id": "ACC-3301",
        "customer_name": "Fabrikam Logistics",
        "address": "44 Harbor Rd, Tacoma, WA 98402, USA",
        "msa_status": "in_review",
        "account_status": "active",
        "segment": "mid-market",
        "buying_context": "Evaluating route-optimization add-ons for peak season.",
    },
}

_CATALOG: tuple[dict[str, Any], ...] = (
    {"sku": "SKU-ANALYTICS-CORE", "name": "Analytics Core Platform", "bundle": "platform", "list_price": 48000, "keywords": ("analytics", "platform", "core")},
    {"sku": "SKU-ANALYTICS-EDGE", "name": "Edge Connector Pack", "bundle": "platform", "list_price": 12000, "keywords": ("analytics", "edge", "connector", "automation")},
    {"sku": "SKU-SUPPORT-PREM", "name": "Premier Support (12 mo)", "bundle": "support", "list_price": 9000, "keywords": ("support", "premier", "service")},
    {"sku": "SKU-ROUTE-OPT", "name": "Route Optimization Add-on", "bundle": "logistics", "list_price": 15000, "keywords": ("route", "optimization", "logistics")},
    {"sku": "SKU-TRAINING-1", "name": "Onboarding & Training", "bundle": "services", "list_price": 6000, "keywords": ("training", "onboarding", "services")},
)

_SKU_INDEX = {entry["sku"]: entry for entry in _CATALOG}
_INCOMPATIBLE_PAIRS = {("SKU-ROUTE-OPT", "SKU-ANALYTICS-EDGE")}
_UNAVAILABLE_SKUS = {"SKU-TRAINING-1"}

_LEGAL_TERMS: dict[str, dict[str, Any]] = {
    "enterprise": {
        "segment": "enterprise",
        "risk_level": "medium",
        "clauses": [
            "Net-45 payment terms.",
            "Standard MSA governs; no bespoke indemnity without legal review.",
            "Auto-renewal with 60-day opt-out.",
        ],
        "approvals_required": ["Deal desk", "Legal (if discount > 20%)"],
    },
    "mid-market": {
        "segment": "mid-market",
        "risk_level": "low",
        "clauses": ["Net-30 payment terms.", "Click-through terms acceptable below $50k."],
        "approvals_required": ["Deal desk"],
    },
}


def _string_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


def crm_get_quote_trigger(opportunity_id: str = "OPP-5001") -> dict[str, Any]:
    """Return CRM trigger state for an opportunity."""

    key = (opportunity_id or "").strip().upper()
    record = _QUOTE_TRIGGERS.get(key)
    if record is None:
        return {"found": False, "opportunity_id": opportunity_id, "known_ids": sorted(_QUOTE_TRIGGERS)}
    return {"found": True, **record}


def crm_get_customer_profile(account_id: str = "ACC-3300") -> dict[str, Any]:
    """Return the enriched CRM customer profile for an account."""

    key = (account_id or "").strip().upper()
    record = _CUSTOMER_PROFILES.get(key)
    if record is None:
        return {"found": False, "account_id": account_id, "known_ids": sorted(_CUSTOMER_PROFILES)}
    return {"found": True, **record}


def product_search_catalog(query: str = "analytics platform") -> dict[str, Any]:
    """Search the product/SKU catalog with a simple keyword match."""

    terms = [term for term in (query or "").lower().replace(",", " ").split() if term]
    scored: list[tuple[int, dict[str, Any]]] = []
    for entry in _CATALOG:
        haystack = " ".join((entry["name"], entry["bundle"], " ".join(entry["keywords"]))).lower()
        score = sum(1 for term in terms if term in haystack)
        if score or not terms:
            scored.append((score, entry))
    scored.sort(key=lambda item: (-item[0], item[1]["sku"]))
    matches = [
        {"sku": e["sku"], "name": e["name"], "bundle": e["bundle"], "list_price": e["list_price"], "match_score": s}
        for s, e in scored
    ]
    return {"query": query, "match_count": len(matches), "matches": matches}


def product_validate_skus(skus: list[str] | None = None) -> dict[str, Any]:
    """Validate SKU compatibility, availability, and completeness."""

    requested = _string_list(skus) or [entry["sku"] for entry in _CATALOG[:2]]
    requested_set = {sku.strip().upper() for sku in requested}
    validated: list[dict[str, Any]] = []
    for sku in requested:
        key = sku.strip().upper()
        known = key in _SKU_INDEX
        available = known and key not in _UNAVAILABLE_SKUS
        compatible = not any(
            {key, other} == set(pair) for pair in _INCOMPATIBLE_PAIRS for other in requested_set
        )
        validated.append(
            {
                "sku": key,
                "known": known,
                "compatible": compatible,
                "available": available,
                "complete": bool(known and available and compatible),
            }
        )
    all_valid = bool(validated) and all(item["complete"] for item in validated)
    return {"requested": requested, "validated": validated, "all_valid": all_valid}


def pricing_calculate_quote(skus: list[str] | None = None, discount_pct: float = 0.0) -> dict[str, Any]:
    """Calculate quote pricing for a set of SKUs."""

    requested = _string_list(skus) or [entry["sku"] for entry in _CATALOG[:2]]
    line_items: list[dict[str, Any]] = []
    subtotal = 0
    for sku in requested:
        key = sku.strip().upper()
        entry = _SKU_INDEX.get(key)
        price = int(entry["list_price"]) if entry else 0
        subtotal += price
        line_items.append({"sku": key, "list_price": price, "in_catalog": entry is not None})
    try:
        pct = float(discount_pct)
    except (TypeError, ValueError):
        pct = 0.0
    pct = max(0.0, min(40.0, pct))
    discount = round(subtotal * pct / 100.0, 2)
    total = round(subtotal - discount, 2)
    return {
        "currency": "USD",
        "line_items": line_items,
        "subtotal": subtotal,
        "discount_pct": pct,
        "discount": discount,
        "total": total,
    }


def legal_evaluate_terms(segment: str = "enterprise", discount_pct: float = 0.0) -> dict[str, Any]:
    """Return legal/contract terms and required approvals for a segment."""

    key = (segment or "").strip().lower()
    terms = _LEGAL_TERMS.get(key, _LEGAL_TERMS["enterprise"])
    try:
        pct = float(discount_pct)
    except (TypeError, ValueError):
        pct = 0.0
    approvals = list(terms["approvals_required"])
    if pct > 20 and "Legal review" not in approvals:
        approvals.append("Legal review (discount over 20%)")
    return {
        "segment": terms["segment"],
        "risk_level": terms["risk_level"],
        "clauses": list(terms["clauses"]),
        "approvals_required": approvals,
    }


def quote_format_package(
    customer_name: str = "Contoso Manufacturing",
    total: float = 0.0,
    skus: list[str] | None = None,
) -> dict[str, Any]:
    """Format the final customer-ready quote package."""

    requested = _string_list(skus)
    fingerprint = "|".join([customer_name, ",".join(requested), f"{float(total or 0.0):.2f}"])
    digest = hashlib.sha256(fingerprint.encode("utf-8")).hexdigest()[:8]
    return {
        "quote_id": f"Q2C-{digest}",
        "format": "pdf",
        "customer_name": customer_name,
        "total": round(float(total or 0.0), 2),
        "skus": [sku.strip().upper() for sku in requested],
        "sections": ["Cover", "Customer & MSA", "Line Items & Pricing", "Terms & Conditions", "Signature"],
        "customer_ready": True,
    }


MCP_TOOL_FUNCTIONS.update(
    {
        "crm_get_quote_trigger": crm_get_quote_trigger,
        "crm_get_customer_profile": crm_get_customer_profile,
        "product_search_catalog": product_search_catalog,
        "product_validate_skus": product_validate_skus,
        "pricing_calculate_quote": pricing_calculate_quote,
        "legal_evaluate_terms": legal_evaluate_terms,
        "quote_format_package": quote_format_package,
    }
)


In [ ]:
from dataclasses import dataclass
from typing import Any

from agent_framework.ollama import OllamaChatClient


DEFAULT_OLLAMA_TEMPERATURE = 0.2
DEFAULT_OLLAMA_NUM_CTX = 8192
DEFAULT_OLLAMA_KEEP_ALIVE = "10m"
DEFAULT_OLLAMA_THINK = False

_UNSUPPORTED_OLLAMA_CHAT_OPTIONS = {
    "allow_multiple_tool_calls",
    "conversation_id",
    "logit_bias",
    "metadata",
    "store",
    "user",
}


@dataclass(frozen=True)
class OllamaAgentConfig:
    model: str
    host: str
    temperature: float
    num_ctx: int
    max_tokens: int
    keep_alive: str
    think: bool

    def default_options(self) -> dict[str, Any]:
        return {
            "temperature": self.temperature,
            "num_ctx": self.num_ctx,
            "max_tokens": self.max_tokens,
            "keep_alive": self.keep_alive,
            "think": self.think,
        }


def parse_env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None or value.strip() == "":
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true or false.")


def build_ollama_config(
    *,
    model: str | None = None,
    host: str | None = None,
    temperature: float | None = None,
    num_ctx: int | None = None,
    max_tokens: int | None = None,
    keep_alive: str | None = None,
    think: bool | None = None,
) -> OllamaAgentConfig:
    return OllamaAgentConfig(
        model=model or os.getenv("OLLAMA_MODEL") or "qwen3:14b",
        host=host or os.getenv("OLLAMA_HOST") or "http://localhost:11434",
        temperature=temperature
        if temperature is not None
        else float(os.getenv("OLLAMA_TEMPERATURE", str(DEFAULT_OLLAMA_TEMPERATURE))),
        num_ctx=num_ctx if num_ctx is not None else int(os.getenv("OLLAMA_NUM_CTX", str(DEFAULT_OLLAMA_NUM_CTX))),
        max_tokens=max_tokens if max_tokens is not None else int(os.getenv("OLLAMA_MAX_TOKENS", "500")),
        keep_alive=keep_alive or os.getenv("OLLAMA_KEEP_ALIVE") or DEFAULT_OLLAMA_KEEP_ALIVE,
        think=think if think is not None else parse_env_bool("OLLAMA_THINK", DEFAULT_OLLAMA_THINK),
    )


class ScenarioOllamaChatClient(OllamaChatClient):
    def _prepare_options(self, messages: Any, options: Any) -> dict[str, Any]:
        filtered = {
            key: value
            for key, value in dict(options).items()
            if key not in _UNSUPPORTED_OLLAMA_CHAT_OPTIONS
        }
        return super()._prepare_options(messages, filtered)


def make_agent(spec: Any, *, config: OllamaAgentConfig | None = None) -> Any:
    resolved = config or build_ollama_config()
    instructions = f"You are {spec.name}. {spec.instructions}"
    tools = tools_for_agent(spec)
    return ScenarioOllamaChatClient(host=resolved.host, model=resolved.model).as_agent(
        name=spec.name,
        description=spec.description,
        instructions=instructions,
        tools=tools or None,
        default_options=resolved.default_options(),
        require_per_service_call_history_persistence=True,
    )


In [ ]:
import json
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class AgentSpec:
    name: str
    description: str
    instructions: str
    mcp_tools: tuple[str, ...] = ()
    mcp_server: str = "enterprise_context"
    code_tools: tuple[str, ...] = ()


@dataclass(frozen=True)
class ScenarioSpec:
    id: str
    pattern: str
    title: str
    learning_goal: str
    when_to_use: str
    sample_input: str
    agents: tuple[AgentSpec, ...]


SCENARIO_DATA = json.loads(
    r"""
{
  "id": "scenario-16-quote-to-cash-group-chat",
  "pattern": "group-chat",
  "title": "Scenario 16: Quote-To-Cash (Group Chat)",
  "learning_goal": "Learn collaborative quote review where agents debate product fit, pricing risk, legal terms, and final quote readiness until they converge.",
  "when_to_use": "Use Group Chat when a visible debate and cross-check between roles improves the quote before it is finalized.",
  "sample_input": "Create a quote for opportunity OPP-5001 (account ACC-3300, Contoso Manufacturing). They want the analytics platform with edge connectors and premier support, with standard enterprise terms.",
  "agents": [
    {
      "name": "QuoteTriggerAgent",
      "description": "Monitors CRM for quote-trigger conditions.",
      "instructions": "Check whether the CRM conditions to create a quote exist. Use crm_get_quote_trigger to report quote-readiness, which trigger conditions are met, and any blockers. Do not invent CRM data.",
      "mcp_tools": [
        "crm_get_quote_trigger"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "note_observation",
        "make_checklist"
      ]
    },
    {
      "name": "CustomerContextAgent",
      "description": "Enriches the customer profile from CRM.",
      "instructions": "Enrich the customer profile. Use crm_get_customer_profile to capture customer name, address, MSA status, account status, segment, and buying context for the quote.",
      "mcp_tools": [
        "crm_get_customer_profile"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "note_observation",
        "compose_summary"
      ]
    },
    {
      "name": "SkuDiscoveryAgent",
      "description": "Finds SKUs, bundles, and catalog entries.",
      "instructions": "Identify candidate SKUs, bundles, and catalog entries that fit the customer's need. Use product_search_catalog and list the matching SKUs with names and list prices.",
      "mcp_tools": [
        "product_search_catalog"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "note_observation",
        "make_checklist"
      ]
    },
    {
      "name": "ProductFitAgent",
      "description": "Validates product compatibility and availability.",
      "instructions": "Validate product compatibility, availability, and SKU completeness. Use product_validate_skus and flag any unknown, unavailable, or incompatible SKUs before pricing.",
      "mcp_tools": [
        "product_validate_skus"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "note_observation",
        "make_checklist"
      ]
    },
    {
      "name": "PricingTermsAgent",
      "description": "Resolves pricing, finance, and legal terms.",
      "instructions": "Resolve pricing, discount, finance, and legal constraints. Use pricing_calculate_quote for totals and legal_evaluate_terms for clauses and required approvals.",
      "mcp_tools": [
        "pricing_calculate_quote",
        "legal_evaluate_terms"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "note_observation",
        "rate_risk"
      ]
    },
    {
      "name": "QuoteGenerationAgent",
      "description": "Generates the final customer-ready quote package.",
      "instructions": "Assemble the final quote package with pricing, SKUs, legal/T&C notes, and a customer-ready format. Use quote_format_package. When you coordinate other specialists, plan the path to a complete quote, delegate the missing context, and stop once the package is ready.",
      "mcp_tools": [
        "quote_format_package"
      ],
      "mcp_server": "quote_to_cash_context",
      "code_tools": [
        "compose_summary",
        "extract_action_items"
      ]
    }
  ]
}
    """
)
AGENTS = tuple(
    AgentSpec(
        name=item["name"],
        description=item["description"],
        instructions=item["instructions"],
        mcp_tools=tuple(item.get("mcp_tools", [])),
        mcp_server=item.get("mcp_server", "enterprise_context"),
        code_tools=tuple(item.get("code_tools", [])),
    )
    for item in SCENARIO_DATA["agents"]
)
SCENARIO = ScenarioSpec(
    id=SCENARIO_DATA["id"],
    pattern=SCENARIO_DATA["pattern"],
    title=SCENARIO_DATA["title"],
    learning_goal=SCENARIO_DATA["learning_goal"],
    when_to_use=SCENARIO_DATA["when_to_use"],
    sample_input=SCENARIO_DATA["sample_input"],
    agents=AGENTS,
)


def tools_for_agent(spec: AgentSpec) -> list[Callable[..., object]]:
    tools: list[Callable[..., object]] = list(resolve_code_tools(effective_code_tools(spec)))
    for tool_name in spec.mcp_tools:
        try:
            tools.append(MCP_TOOL_FUNCTIONS[tool_name])
        except KeyError as exc:
            raise ValueError(f"Missing inlined tool '{tool_name}' for {spec.name}.") from exc
    return tools


def scenario_summary(scenario: ScenarioSpec) -> dict[str, str]:
    return {
        "id": scenario.id,
        "title": scenario.title,
        "pattern": scenario.pattern,
        "learning_goal": scenario.learning_goal,
        "when_to_use": scenario.when_to_use,
        "sample": getattr(scenario, "sample_input"),
    }


def coded_agent_tool_map(scenario: ScenarioSpec) -> list[dict[str, Any]]:
    return [
        {
            "agent": spec.name,
            "code_tools": list(effective_code_tools(spec)),
            "mcp_tools": list(spec.mcp_tools),
            "mcp_server": spec.mcp_server if spec.mcp_tools else None,
        }
        for spec in scenario.agents
    ]


def mcp_tool_context(scenario: ScenarioSpec) -> dict[str, Any]:
    tools_by_agent = {spec.name: list(spec.mcp_tools) for spec in scenario.agents if spec.mcp_tools}
    all_tools_used = sorted({tool for spec in scenario.agents for tool in spec.mcp_tools})
    return {
        "uses_mcp": bool(all_tools_used),
        "tools_by_agent": tools_by_agent,
        "all_tools_used": all_tools_used,
    }


MAX_TOKENS = 500 if SCENARIO.pattern == "magentic" else 250
RESPONSES_PAYLOAD = {"input": SCENARIO.sample_input, "stream": False}
SAMPLE_PROMPT = SCENARIO.sample_input

print(json.dumps(scenario_summary(SCENARIO), indent=2))
print(json.dumps(coded_agent_tool_map(SCENARIO), indent=2))
if mcp_tool_context(SCENARIO)["uses_mcp"]:
    print(json.dumps(mcp_tool_context(SCENARIO), indent=2))


In [ ]:
import re
from collections.abc import Mapping
from typing import Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)


_TRANSCRIPT_KEY = "transcript"
_STOPWORDS = {"agent", "specialist", "the", "and", "for", "with", "that", "from", "into"}


def make_request(text: str) -> AgentExecutorRequest:
    return AgentExecutorRequest(messages=[Message(role="user", contents=[text])])


def response_text(response: AgentExecutorResponse) -> str:
    agent_response = getattr(response, "agent_response", None)
    return (getattr(agent_response, "text", None) or "").strip()


def _append_transcript(ctx: WorkflowContext[Any], author: str, text: str) -> list[str]:
    transcript = list(ctx.get_state(_TRANSCRIPT_KEY) or [])
    transcript.append(f"[{author}] {text}")
    ctx.set_state(_TRANSCRIPT_KEY, transcript)
    return transcript


class PromptDispatchExecutor(Executor):
    @handler
    async def dispatch(self, prompt: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        ctx.set_state("prompt", prompt)
        ctx.set_state(_TRANSCRIPT_KEY, [])
        await ctx.send_message(make_request(prompt))


class StageGateExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def gate(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        prompt = ctx.get_state("prompt") or ""
        carried = "\n".join(transcript)
        await ctx.send_message(make_request(f"Original request:\n{prompt}\n\nWork so far:\n{carried}"))


class SequentialOutputExecutor(Executor):
    def __init__(self, id: str, *, stage_name: str) -> None:
        super().__init__(id=id)
        self._stage_name = stage_name

    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        transcript = _append_transcript(ctx, self._stage_name, response_text(response))
        await ctx.yield_output("\n\n".join(transcript))


class ConcurrentAggregatorExecutor(Executor):
    def __init__(self, id: str, *, agent_names: list[str]) -> None:
        super().__init__(id=id)
        self._agent_names = agent_names

    @handler
    async def aggregate(self, responses: list[AgentExecutorResponse], ctx: WorkflowContext[Never, str]) -> None:
        labelled: list[str] = []
        for index, response in enumerate(responses):
            name = self._agent_names[index] if index < len(self._agent_names) else f"agent{index + 1}"
            labelled.append(f"[{name}]\n{response_text(response)}")
        await ctx.yield_output("\n\n".join(labelled))


class HandoffRouterExecutor(Executor):
    def __init__(self, id: str, *, routes: dict[str, tuple[str, ...]], default_route: str) -> None:
        super().__init__(id=id)
        self._routes = routes
        self._default_route = default_route

    def choose(self, text: str) -> str:
        lowered = text.lower()
        best_route, best_hits = self._default_route, 0
        for route, keywords in self._routes.items():
            hits = sum(1 for keyword in keywords if keyword in lowered)
            if hits > best_hits:
                best_route, best_hits = route, hits
        return best_route

    @handler
    async def route(self, response: AgentExecutorResponse, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
        triage_text = response_text(response)
        chosen = self.choose(triage_text)
        ctx.set_state("route", chosen)
        prompt = ctx.get_state("prompt") or ""
        await ctx.send_message(
            make_request(f"Triage routed this to you.\nRequest:\n{prompt}\n\nTriage notes:\n{triage_text}"),
            target_id=chosen,
        )


class HandoffOutputExecutor(Executor):
    @handler
    async def finish(self, response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
        route = ctx.get_state("route") or "specialist"
        await ctx.yield_output(f"[routed to {route}]\n{response_text(response)}")


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def _agents_for(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> list[Any]:
    return [make_agent(spec, config=config) for spec in scenario.agents]


def _agent_executor(spec_index: int, scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> AgentExecutor:
    spec = scenario.agents[spec_index]
    return AgentExecutor(make_agent(spec, config=config), id=_slug(spec.name))


def _route_keywords(spec: AgentSpec) -> tuple[str, ...]:
    tokens = re.findall(r"[a-z]+", f"{spec.name} {spec.description}".lower())
    keywords = [token for token in tokens if len(token) > 3 and token not in _STOPWORDS]
    return tuple(dict.fromkeys(keywords))[:6]


def build_sequential_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    agents = [_agent_executor(i, scenario, config=config) for i in range(len(scenario.agents))]
    dispatch = PromptDispatchExecutor(id="dispatch")
    output = SequentialOutputExecutor(id="final_output", stage_name=scenario.agents[-1].name)
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, agents[0])
    for index in range(len(agents) - 1):
        gate = StageGateExecutor(id=f"gate_{index}", stage_name=scenario.agents[index].name)
        builder.add_edge(agents[index], gate)
        builder.add_edge(gate, agents[index + 1])
    builder.add_edge(agents[-1], output)
    return builder.build()


def build_concurrent_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    agents = [_agent_executor(i, scenario, config=config) for i in range(len(scenario.agents))]
    dispatch = PromptDispatchExecutor(id="dispatch")
    aggregator = ConcurrentAggregatorExecutor(id="aggregator", agent_names=[spec.name for spec in scenario.agents])
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[aggregator])
    builder.add_fan_out_edges(dispatch, agents)
    builder.add_fan_in_edges(agents, aggregator)
    return builder.build()


def build_handoff_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    triage = _agent_executor(0, scenario, config=config)
    specialists = [_agent_executor(i, scenario, config=config) for i in range(1, len(scenario.agents))]
    specialist_ids = [_slug(scenario.agents[i].name) for i in range(1, len(scenario.agents))]
    routes = {
        specialist_ids[i - 1]: _route_keywords(scenario.agents[i])
        for i in range(1, len(scenario.agents))
    }
    dispatch = PromptDispatchExecutor(id="dispatch")
    router = HandoffRouterExecutor(id="router", routes=routes, default_route=specialist_ids[0])
    output = HandoffOutputExecutor(id="final_output")
    builder = WorkflowBuilder(start_executor=dispatch, output_from=[output])
    builder.add_edge(dispatch, triage)
    builder.add_edge(triage, router)
    for specialist in specialists:
        builder.add_edge(router, specialist)
        builder.add_edge(specialist, output)
    return builder.build()


def build_group_chat_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import GroupChatBuilder

    participants = _agents_for(scenario, config=config)

    def round_robin_selector(state: Any) -> str:
        participant_names = list(state.participants.keys())
        return participant_names[state.current_round % len(participant_names)]

    def stop_after_council(messages: list[Any]) -> bool:
        assistant_messages = [m for m in messages if getattr(m, "role", None) == "assistant"]
        if len(assistant_messages) >= 7:
            return True
        last_text = getattr(messages[-1], "text", "").lower() if messages else ""
        return "approved" in last_text and "recommendation" in last_text

    return GroupChatBuilder(
        participants=participants,
        selection_func=round_robin_selector,
        termination_condition=stop_after_council,
        intermediate_output_from=participants,
    ).build()


def build_magentic_workflow(scenario: ScenarioSpec, *, config: OllamaAgentConfig) -> Any:
    from agent_framework.orchestrations import MagenticBuilder

    agents = _agents_for(scenario, config=config)
    manager_agent = agents[0]
    participants = agents[1:]
    return MagenticBuilder(
        participants=participants,
        intermediate_output_from=participants,
        manager_agent=manager_agent,
        max_round_count=10,
        max_stall_count=3,
        max_reset_count=2,
    ).build()


def build_workflow(
    scenario: ScenarioSpec = SCENARIO,
    *,
    max_tokens: int | None = None,
    **config_overrides: Any,
) -> Any:
    config = build_ollama_config(max_tokens=max_tokens or MAX_TOKENS, **config_overrides)
    builders = {
        "sequential": build_sequential_workflow,
        "concurrent": build_concurrent_workflow,
        "handoff": build_handoff_workflow,
        "group-chat": build_group_chat_workflow,
        "magentic": build_magentic_workflow,
    }
    return builders[scenario.pattern](scenario, config=config)


def workflow_result_to_text(result: Any) -> str:
    outputs = result.get_outputs() if hasattr(result, "get_outputs") else result
    intermediate = result.get_intermediate_outputs() if hasattr(result, "get_intermediate_outputs") else []
    if not outputs:
        intermediate_text = join_readable_outputs(intermediate)
        return intermediate_text or "No workflow output was produced."
    output_text = join_readable_outputs(outputs)
    if intermediate and should_use_intermediate_outputs(output_text):
        intermediate_text = join_readable_outputs(intermediate)
        if intermediate_text:
            return intermediate_text
    return output_text or "No readable workflow text was produced."


def join_readable_outputs(outputs: Any) -> str:
    return "\n\n".join(text for output in outputs if (text := agent_response_to_text(output)))


def should_use_intermediate_outputs(output_text: str) -> bool:
    normalized = output_text.strip().lower()
    if not normalized:
        return True
    if len(normalized) >= 160:
        return False
    markers = ("termination condition", "maximum reset count", "maximum stall count", "workflow terminated")
    return any(marker in normalized for marker in markers)


def agent_response_to_text(value: Any) -> str:
    text = extract_text(value)
    return text or "No readable workflow text was produced."


def extract_text(value: Any, seen: set[int] | None = None) -> str:
    if value is None:
        return ""
    if seen is None:
        seen = set()
    value_id = id(value)
    if value_id in seen:
        return ""
    seen.add(value_id)
    if isinstance(value, str):
        return "" if is_object_repr(value) else value
    text = getattr(value, "text", None)
    if isinstance(text, str) and text and not is_object_repr(text):
        return text
    messages = getattr(value, "messages", None)
    if messages:
        parts: list[str] = []
        for message in messages:
            author = getattr(message, "author_name", None) or getattr(message, "role", None) or "assistant"
            message_text = extract_text(message, seen)
            if message_text:
                parts.append(f"[{author}] {message_text}")
        if parts:
            return "\n".join(parts)
    contents = getattr(value, "contents", None)
    if contents:
        return "\n".join(part for content in contents if (part := extract_text(content, seen)))
    items = getattr(value, "items", None)
    if items and not callable(items):
        return "\n".join(part for item in items if (part := extract_text(item, seen)))
    result = getattr(value, "result", None)
    if result is not None:
        return extract_text(result, seen)
    if isinstance(value, Mapping):
        parts = [extract_text(value.get(key), seen) for key in ("text", "content", "message", "summary", "result")]
        return "\n".join(part for part in parts if part)
    if isinstance(value, Sequence) and not isinstance(value, (bytes, bytearray, str)):
        return "\n".join(part for item in value if (part := extract_text(item, seen)))
    fallback = str(value)
    return "" if is_object_repr(fallback) else fallback


def is_object_repr(value: str) -> bool:
    return value.startswith("<") and " object at 0x" in value and value.endswith(">")


## Flow Diagram

The diagram below shows 6 participants in a round-robin loop with a coded termination function attached to the Responses API /responses.
Solid arrows are orchestration edges. Dashed arrows (`-.->`) are tool calls.
MCP tool nodes use a stadium shape; code tool nodes use a parallelogram.


In [ ]:
import base64
import html
from dataclasses import dataclass

from IPython.display import HTML, display


@dataclass(frozen=True)
class ScenarioFlowDiagram:
    title: str
    mermaid: str
    image_url: str


def scenario_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _diagram_source(scenario, api_boundary="Responses API /responses", input_label="OpenAI-style input")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_scenario_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = scenario_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _diagram_source(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    if scenario.pattern == "sequential":
        return _sequential_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "concurrent":
        return _concurrent_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "handoff":
        return _handoff_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "group-chat":
        return _group_chat_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    if scenario.pattern == "magentic":
        return _magentic_diagram(scenario, api_boundary=api_boundary, input_label=input_label)
    raise ValueError(f"Unsupported pattern '{scenario.pattern}'.")


def _sequential_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    previous = "orchestrator"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} -->|stage {index}| {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    lines.extend(_code_tool_links(pairs))
    return "\n".join(lines)


def _concurrent_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> fanout{Fan out same request}")
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    fanout --> {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> aggregate{{Aggregate findings}}")
        pairs.append((agent, node))
    lines.append("    aggregate --> output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    lines.extend(_code_tool_links(pairs))
    return "\n".join(lines)


def _handoff_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    triage, *specialists = scenario.agents
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append(f"    orchestrator --> triage[{_label(triage.name)}]")
    lines.append("    triage --> decision{Ownership decision}")
    pairs: list[tuple[AgentSpec, str]] = [(triage, "triage")]
    for index, agent in enumerate(specialists, start=1):
        node = f"specialist{index}"
        lines.append(f"    decision -->|handoff| {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> output[Responses output]")
        pairs.append((agent, node))
    lines.extend(_mcp_tool_links(pairs))
    lines.extend(_code_tool_links(pairs))
    return "\n".join(lines)


def _group_chat_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append("    orchestrator --> selector{Round-robin selector}")
    previous = "selector"
    pairs: list[tuple[AgentSpec, str]] = []
    for index, agent in enumerate(scenario.agents, start=1):
        node = f"agent{index}"
        lines.append(f"    {previous} --> {node}[{_label(agent.name)}]")
        previous = node
        pairs.append((agent, node))
    lines.append(f"    {previous} --> stop{{Termination condition}}")
    lines.append("    stop -->|continue| selector")
    lines.append("    stop -->|done| output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    lines.extend(_code_tool_links(pairs))
    return "\n".join(lines)


def _magentic_diagram(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> str:
    manager, *specialists = scenario.agents
    lines = _header(scenario, api_boundary=api_boundary, input_label=input_label)
    lines.append(f"    orchestrator --> manager[{_label(manager.name)}]")
    lines.append("    manager --> plan{Plan and delegate}")
    pairs: list[tuple[AgentSpec, str]] = [(manager, "manager")]
    for index, agent in enumerate(specialists, start=1):
        node = f"specialist{index}"
        lines.append(f"    plan --> {node}[{_label(agent.name)}]")
        lines.append(f"    {node} --> progress{{Progress ledger}}")
        pairs.append((agent, node))
    lines.append("    progress -->|replan| manager")
    lines.append("    progress -->|complete or stop| output[Responses output]")
    lines.extend(_mcp_tool_links(pairs))
    lines.extend(_code_tool_links(pairs))
    return "\n".join(lines)


def _header(scenario: ScenarioSpec, *, api_boundary: str, input_label: str) -> list[str]:
    return [
        "flowchart TD",
        f"    client[{_label(input_label)}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
    ]


def _mcp_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in agent.mcp_tools:
            lines.append(f"    {node_id} -.->|mcp tool| tool_{tool}([{_label(tool)}])")
    return lines


def _code_tool_links(pairs: list[tuple[AgentSpec, str]]) -> list[str]:
    lines: list[str] = []
    for agent, node_id in pairs:
        for tool in effective_code_tools(agent):
            lines.append(f"    {node_id} -.->|code tool| ctool_{tool}[/{_label(tool)}/]")
    return lines


def quote_to_cash_flow_diagram(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    mermaid = _quote_to_cash_source(scenario, api_boundary="Responses API /responses")
    return ScenarioFlowDiagram(
        title=f"{scenario.title} Quote-To-Cash Flow",
        mermaid=mermaid,
        image_url=_mermaid_image_url(mermaid),
    )


def display_quote_to_cash_flow(scenario: ScenarioSpec) -> ScenarioFlowDiagram:
    diagram = quote_to_cash_flow_diagram(scenario)
    display(
        HTML(
            '<figure style="margin: 0">'
            f'<img src="{html.escape(diagram.image_url)}" alt="{html.escape(diagram.title)}" '
            'style="max-width: 100%; height: auto;" />'
            f'<figcaption style="font-size: 0.9em; color: #555;">{html.escape(diagram.title)}</figcaption>'
            "</figure>"
        )
    )
    return diagram


def _quote_to_cash_source(scenario: ScenarioSpec, *, api_boundary: str) -> str:
    names = {agent.name for agent in scenario.agents}

    def node(role: str) -> str:
        return role if role in names else next(iter(names))

    lines = [
        "flowchart TD",
        f"    client[{_label('Quote request begins in CRM')}] --> api[{_label(api_boundary)}]",
        f"    api --> scenario[{_label('Scenario: ' + scenario.id)}]",
        f"    scenario --> orchestrator{{{_label(scenario.pattern + ' orchestration')}}}",
        f"    orchestrator --> crm[{_label('CRM system')}]",
        f"    crm -->|wave 1| trigger[{_label(node('QuoteTriggerAgent'))}]",
        f"    crm -->|wave 1| customer[{_label(node('CustomerContextAgent'))}]",
        f"    trigger --> store1[({_label('Orchestration store: customer context')})]",
        "    customer --> store1",
        f"    store1 -. {_label('deallocate wave 1')} .-> freed1(({_label('agents released')}))",
        f"    store1 --> product[{_label('Product / SKU system')}]",
        f"    product -->|wave 2| sku[{_label(node('SkuDiscoveryAgent'))}]",
        f"    product -->|wave 2| fit[{_label(node('ProductFitAgent'))}]",
        f"    sku --> store2[({_label('Orchestration store: product context')})]",
        "    fit --> store2",
        f"    store2 -. {_label('deallocate wave 2')} .-> freed2(({_label('agents released')}))",
        f"    store2 --> pricingsys[{_label('Pricing / finance / legal system')}]",
        f"    pricingsys -->|wave 3| pricing[{_label(node('PricingTermsAgent'))}]",
        f"    pricing --> generation[{_label(node('QuoteGenerationAgent'))}]",
        f"    generation --> quote[/{_label('Final quote package')}/]",
    ]
    return "\n".join(lines)


def _label(value: str) -> str:
    return value.replace('"', "'")


def _mermaid_image_url(mermaid: str) -> str:
    encoded = base64.urlsafe_b64encode(mermaid.encode("utf-8")).decode("ascii").rstrip("=")
    return f"https://mermaid.ink/img/{encoded}"


flow_diagram = display_scenario_flow(SCENARIO)
quote_to_cash_diagram = display_quote_to_cash_flow(SCENARIO)
print(flow_diagram.mermaid)


## Live Run

Participants speak in round-robin order. The termination function fires when an approved recommendation appears or after seven assistant turns. Intermediate outputs from each participant are surfaced alongside the final transcript.

> **Instructor note:** `qwen3:14b` runs with `think: False` by default (extended reasoning off).
> Set `OLLAMA_THINK=true` before the environment cell to enable chain-of-thought reasoning --
> useful when debugging unexpected routing decisions or tool call sequences.


In [ ]:
workflow = build_workflow(SCENARIO, max_tokens=MAX_TOKENS)
result = await workflow.run(SAMPLE_PROMPT)
output_text = workflow_result_to_text(result)

if not output_text.strip():
    raise RuntimeError("Workflow completed but produced no readable text.")

print(output_text)


## What to Inspect

Read the transcript chronologically. Later turns should respond to earlier critiques rather than restarting the discussion. The termination function fires on 'approved' plus 'recommendation' in the same message, or after seven assistant turns -- check which condition fired and why.


## Experiments

- Change `RESPONSES_PAYLOAD['input']` and rerun the live cell.
- Override `OLLAMA_MODEL` or `OLLAMA_HOST` before the environment cell to target a different local Ollama setup.
- Inspect `coded_agent_tool_map(SCENARIO)` and remove one tool from an agent to see how the answer changes.
- Lower `MAX_TOKENS` for faster runs or raise it when group-chat needs more room.
